
# Manejo de Archivos: Lectura/Escritura de archivos y Operaciones con Cadenas en Python



## Índice
1. Conceptos base y buenas prácticas de I/O
2. `open()`: modos, buffering, newline, encoding, errores
3. Texto vs binario: `str` y `bytes`, conversiones y errores
4. Iteración eficiente sobre archivos grandes (memory‑safe)
5. Escritura segura y atómica (patrones con `tempfile`)
6. Rutas modernas con `pathlib` (vs `os`), globbing y utilidades
7. Streams en memoria con `io.StringIO` y `io.BytesIO`
8. Archivos comprimidos: `gzip`, `bz2`, `lzma`
9. CSV y TSV con `csv` (DictReader/Writer)
10. JSON: lectura, escritura y pretty printing
11. Serialización binaria con `pickle` (y advertencias)
12. Descarga/lectura de múltiples archivos con patrones (globs)
13. Manejo robusto de excepciones y registro (logging básico)
14. Cadenas: repaso intensivo de API (`split`, `join`, `partition`, etc.)
15. F‑strings, `str.format`, `Template`
16. Búsqueda y reemplazo con `re` (regex) + grupos y backrefs
17. Limpieza y normalización de texto (`unicodedata`, `casefold`)
18. Mapas de traducción y tablas (`str.translate`, `maketrans`)
19. Envoltura y truncado de texto (`textwrap`)
20. Rendimiento: `join` vs concatenación, `io` buffered, slicing
21. Mini‑proyecto: ETL de logs (mezcla de I/O + cadenas + regex)
22. Recursos finales



## 1. Conceptos base y buenas prácticas de I/O

- Usa **context managers** (`with open(...) as f:`) para cerrar archivos automáticamente.
- Preferir **`pathlib.Path`** para rutas: facilita portabilidad y operaciones comunes.
- Define **`encoding='utf-8'`** explícitamente al leer/escribir texto.
- Controla **buffering** y **newline** según el caso (rendimiento y consistencia).
- Para archivos grandes: **iterar por líneas** o **en bloques** (chunks).
- Para escritura robusta: **escrituras atómicas** con `tempfile` + `Path.replace()`.
- Validar entradas/salidas y capturar **excepciones** (`try/except`) con mensajes claros.


In [1]:

# 2. open(): modos, buffering, newline, encoding, errores
from pathlib import Path

p = Path('ejemplo_texto.txt')

# Escritura en texto con opciones explícitas
with p.open(mode='w', encoding='utf-8', newline='') as f:
    # newline='' evita traducciones de fin de línea en algunos entornos
    f.write('Línea 1 \n')
    f.write('Línea 2 con acento: café\n')

# Lectura con manejo de errores de codificación
try:
    with p.open(mode='r', encoding='utf-8', errors='strict') as f:
        contenido = f.read()
        print('Contenido leído:\n', contenido)
except UnicodeDecodeError as e:
    print('Error de decodificación:', e)

# Apertura en modo append (agrega al final)
with p.open('a', encoding='utf-8') as f:
    f.write('Línea 3 (append)\n')

print('--- Después de append ---')
print(p.read_text(encoding='utf-8'))


Contenido leído:
 Línea 1 
Línea 2 con acento: café

--- Después de append ---
Línea 1 
Línea 2 con acento: café
Línea 3 (append)



In [2]:

# 3. Texto vs binario: str y bytes
from pathlib import Path

pb = Path('ejemplo_binario.bin')

# Escribir bytes (binario)
data = bytes([0, 1, 2, 255])  # cuatro bytes
with pb.open('wb') as f:
    f.write(data)

# Leer bytes y mostrar su representación
with pb.open('rb') as f:
    b = f.read()
    print('Bytes leídos:', b, ' -> lista:', list(b))

# Convertir entre str y bytes con encode/decode
s = "Hola ß café"
b = s.encode('utf-8', errors='strict')   # str -> bytes
print('UTF-8 bytes:', b)
print('De vuelta a str:', b.decode('utf-8'))


Bytes leídos: b'\x00\x01\x02\xff'  -> lista: [0, 1, 2, 255]
UTF-8 bytes: b'Hola \xc3\x9f caf\xc3\xa9'
De vuelta a str: Hola ß café


In [3]:
# 4. Iteración eficiente sobre archivos grandes
from pathlib import Path
import random

p = Path('grande.txt')

# Generar un archivo realmente "grande" (100k líneas) de forma reproducible (seed fija)
random.seed(42)
nombres = ['Ana', 'Luis', 'Sebastian', 'Felipe', 'Santiago', 'Pablo', 'Maria', 'Carlos', 'Elena', 'Zoe']
with p.open('w', encoding='utf-8') as f:
    for i in range(100_000):
        nombre = random.choice(nombres)
        nota = random.randint(1, 5)
        f.write(f'{i},{nombre},{nota}\n')  # formato simple: id,nombre,nota

print('Archivo generado:', p, '-', sum(1 for _ in p.open(encoding='utf-8')), 'líneas')

Archivo generado: grande.txt - 100000 líneas


In [4]:
# Iterar el archivo sin cargarlo completo en memoria y mostrar las primeras líneas parseadas
with p.open('r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        id_, nombre, nota = line.strip().split(',')
        print(f'id={id_} nombre={nombre} nota={nota}')

id=0 nombre=Luis nota=1
id=1 nombre=Santiago nota=2
id=2 nombre=Felipe nota=2
id=3 nombre=Luis nota=5
id=4 nombre=Luis nota=5


In [5]:
# Iteración línea por línea (streaming, memoria O(1) por línea)
# no uses f.readlines() en archivos de cientos de miles de líneas: eso carga todo en RAM de una vez.
suma = 0
cont = 0
with p.open('r', encoding='utf-8') as f:
    for line in f:
        _id, _nombre, nota = line.strip().split(',')
        suma += int(nota)
        cont += 1
print('Promedio de notas:', suma / cont, '  (sobre', cont, 'registros)')

# Lectura por bloques (chunks) si se requiere control fino del buffering
suma2 = 0
with p.open('r', encoding='utf-8') as f:
    while True:
        chunk = f.read(1_000_000)  # ~1MB
        if not chunk:
            break
        suma2 += chunk.count('\n')
print('Número de líneas (aprox):', suma2)

Promedio de notas: 3.00051   (sobre 100000 registros)


Número de líneas (aprox): 100000


In [6]:

# 5. Escritura segura y 'atómica' con tempfile + replace
# Útil para evitar archivos corruptos si el proceso se interrumpe durante la escritura.
import tempfile
from pathlib import Path

dest = Path('salida_segura.txt')
contenido = 'Resultado crítico\nLínea final\n'

with tempfile.NamedTemporaryFile('w', encoding='utf-8', delete=False, dir=dest.parent) as tmp:
    tmp.write(contenido)
    temp_path = Path(tmp.name)

# Reemplaza de manera atómica (en el mismo disco)
temp_path.replace(dest)
print('Escritura atómica completada en:', dest)
print(dest.read_text(encoding='utf-8'))


Escritura atómica completada en: salida_segura.txt
Resultado crítico
Línea final



In [7]:

# 6. Rutas con pathlib: crear, listar, filtrar, mover/copiar
from pathlib import Path
import shutil

base = Path('demo_pathlib')
(base / 'subdir').mkdir(parents=True, exist_ok=True)

# Crear archivos de ejemplo
for i in range(3):
    (base / f'archivo_{i}.txt').write_text(f'contenido_{i}\n', encoding='utf-8')
(base / 'subdir' / 'data.csv').write_text('id,valor\n1,100\n2,200\n', encoding='utf-8')

# Listar .txt
txts = list(base.glob('*.txt'))
print('TXT:', txts)

# Buscar recursivamente CSV
csvs = list(base.rglob('*.csv'))
print('CSV recursivos:', csvs)

# Mover/copy
dest_dir = base / 'backup'
dest_dir.mkdir(exist_ok=True)
for p in txts:
    shutil.copy2(p, dest_dir / p.name)  # copia preservando metadatos
print('Copiados a', dest_dir)


TXT: [PosixPath('demo_pathlib/archivo_1.txt'), PosixPath('demo_pathlib/archivo_2.txt'), PosixPath('demo_pathlib/archivo_0.txt')]
CSV recursivos: [PosixPath('demo_pathlib/subdir/data.csv')]
Copiados a demo_pathlib/backup


In [8]:

# 7. Streams en memoria con io.StringIO / io.BytesIO
from io import StringIO, BytesIO

# StringIO: se comporta como archivo de texto en RAM
buf = StringIO()
buf.write('Linea A\n')
buf.write('Linea B\n')
buf.seek(0)
print('StringIO ->')
print(buf.read())

# BytesIO: como archivo binario en RAM
bbuf = BytesIO()
bbuf.write(b'\x00\x01\x02')
bbuf.seek(0)
print('BytesIO ->', list(bbuf.read()))


StringIO ->
Linea A
Linea B

BytesIO -> [0, 1, 2]


In [9]:

# 8. Archivos comprimidos: gzip, bz2, lzma
import gzip, bz2, lzma
from pathlib import Path

texto = 'Hola compresión!\n' * 1000

# gzip
gz_path = Path('ejemplo.txt.gz')
with gzip.open(gz_path, 'wt', encoding='utf-8') as f:
    f.write(texto)

with gzip.open(gz_path, 'rt', encoding='utf-8') as f:
    print('GZIP primeras 2 líneas:', [next(f).strip(), next(f).strip()])

# bz2
bz2_path = Path('ejemplo.txt.bz2')
with bz2.open(bz2_path, 'wt', encoding='utf-8') as f:
    f.write(texto)

# lzma/xz
xz_path = Path('ejemplo.txt.xz')
with lzma.open(xz_path, 'wt', encoding='utf-8') as f:
    f.write(texto)
print('Creado gzip/bz2/xz en:', gz_path, bz2_path, xz_path)


GZIP primeras 2 líneas: ['Hola compresión!', 'Hola compresión!']


Creado gzip/bz2/xz en: ejemplo.txt.gz ejemplo.txt.bz2 ejemplo.txt.xz


In [10]:
# 9. CSV y TSV con csv
import csv
from pathlib import Path

csv_path = Path('ejemplo.csv')

rows = [
    {'id': 1, 'nombre': 'Ana', 'nota': 4.5},
    {'id': 2, 'nombre': 'Luis', 'nota': 3.8},
    {'id': 3, 'nombre': 'Zoe', 'nota': 4.9},
]

# Escritura con DictWriter
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['id', 'nombre', 'nota'])
    writer.writeheader()
    writer.writerows(rows)

print('CSV escrito en:', csv_path)

CSV escrito en: ejemplo.csv


In [11]:
# Lectura con DictReader (csv siempre entrega strings: hay que castear los tipos que necesites)
with csv_path.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    datos = [{'id': int(r['id']), 'nombre': r['nombre'], 'nota': float(r['nota'])} for r in reader]

print('Leído desde CSV:', datos)

Leído desde CSV: [{'id': 1, 'nombre': 'Ana', 'nota': 4.5}, {'id': 2, 'nombre': 'Luis', 'nota': 3.8}, {'id': 3, 'nombre': 'Zoe', 'nota': 4.9}]


In [12]:
# Búsqueda por id ya con los tipos correctos (sin necesitar input())
id_buscado = 2
encontrado = next((d for d in datos if d['id'] == id_buscado), None)
if encontrado:
    print(f"id={encontrado['id']} -> nombre={encontrado['nombre']}, nota={encontrado['nota']}")
else:
    print('id no encontrado:', id_buscado)

id=2 -> nombre=Luis, nota=3.8


In [13]:

# 10. JSON: lectura/escritura/pretty
import json
from pathlib import Path

jpath = Path('ejemplo.json')
obj = {
    'curso': 'I/O Avanzado',
    'alumnos': ['Ana', 'Luis', 'Zoé'],
    'config': {'encoding': 'utf-8', 'newline': ''},
}

with jpath.open('w', encoding='utf-8') as f:
    json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)

# Cargar
with jpath.open('r', encoding='utf-8') as f:
    cargado = json.load(f)

print('JSON cargado:', cargado)


JSON cargado: {'alumnos': ['Ana', 'Luis', 'Zoé'], 'config': {'encoding': 'utf-8', 'newline': ''}, 'curso': 'I/O Avanzado'}


In [14]:
import json
obj = {"nombre": "Zoe", "curso": "I/O Avanzado", "nota": 4.9}

# Compacto, ASCII escapado:
print(json.dumps(obj))
# -> {"nombre": "Zoe", "curso": "I/O Avanzado", "nota": 4.9}

# Legible, Unicode, claves ordenadas:
print(json.dumps(obj, ensure_ascii=False, indent=2, sort_keys=True))
# -> {
#      "curso": "I/O Avanzado",
#      "nombre": "Zoe",
#      "nota": 4.9
#    }

{"nombre": "Zoe", "curso": "I/O Avanzado", "nota": 4.9}
{
  "curso": "I/O Avanzado",
  "nombre": "Zoe",
  "nota": 4.9
}


In [15]:

# 11. Serialización binaria con pickle (¡cuidado con datos no confiables!)
import pickle
from pathlib import Path

pkl = Path('ejemplo.pkl')
modelo_simulado = {'w': [1.0, 2.0, 3.0], 'bias': 0.1}

# Guardar
with pkl.open('wb') as f:
    pickle.dump(modelo_simulado, f, protocol=pickle.HIGHEST_PROTOCOL)

# Cargar
with pkl.open('rb') as f:
    m = pickle.load(f)

print('Pickle cargado:', m)

# ADVERTENCIA: nunca uses pickle.load() en archivos de orígenes no confiables (riesgo de ejecución de código).


Pickle cargado: {'w': [1.0, 2.0, 3.0], 'bias': 0.1}


In [16]:

# 12. Procesamiento batch con globs (patrones)
from pathlib import Path

base = Path('lotes')
base.mkdir(exist_ok=True)
for i in range(5):
    (base / f'parte_{i}.txt').write_text(f'linea_{i}\n', encoding='utf-8')

# Leer todos los archivos 'parte_*.txt' y consolidar
salida = Path('consolidado.txt')
with salida.open('w', encoding='utf-8') as out:
    for archivo in sorted(base.glob('parte_*.txt')):
        contenido = archivo.read_text(encoding='utf-8')
        out.write(f'# {archivo.name}\n{contenido}')

print('Consolidado ->', salida)
print(salida.read_text(encoding='utf-8')[:120], '...')


Consolidado -> consolidado.txt
# parte_0.txt
linea_0
# parte_1.txt
linea_1
# parte_2.txt
linea_2
# parte_3.txt
linea_3
# parte_4.txt
linea_4
 ...


In [17]:

# 13. Manejo robusto de excepciones y logging
import logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(message)s')

def leer_seguro(path: Path, encoding='utf-8'):
    try:
        texto = path.read_text(encoding=encoding, errors='strict')
        logging.info('Leído %s (%d bytes)', path, len(texto))
        return texto
    except FileNotFoundError:
        logging.error('No existe: %s', path)
    except UnicodeDecodeError as e:
        logging.error('Error de decodificación en %s: %s', path, e)
    return ''

# Prueba
print(leer_seguro(Path('no_existe.txt')))


ERROR:No existe: no_existe.txt


In [18]:

# 14. API de cadenas: split, rsplit, partition, rpartition, find, replace...
s = "  Python, I/O avanzado; cadenas;  rendimiento  "
print('strip:', s.strip())
print('split por ;:', [p.strip() for p in s.split(';')])
print('rsplit maxsplit=1:', s.rsplit(';', maxsplit=1))
print('partition por coma:', s.partition(','))  # (antes, separador, después)
print('replace:', s.replace('avanzado', 'PRO'))
print('startswith("  Py"):', s.startswith('  Py'))
print('endswith("miento  "):', s.endswith('miento  '))
print('find("I/O"):', s.find('I/O'))
print('count("a"):', s.count('a'))
print('join:', ' | '.join(['a','b','c']))
print('zfill(5):', '42'.zfill(5))
print('center(20, "-"):', 'TÍTULO'.center(20, '-'))


strip: Python, I/O avanzado; cadenas;  rendimiento
split por ;: ['Python, I/O avanzado', 'cadenas', 'rendimiento']
rsplit maxsplit=1: ['  Python, I/O avanzado; cadenas', '  rendimiento  ']
partition por coma: ('  Python', ',', ' I/O avanzado; cadenas;  rendimiento  ')
replace:   Python, I/O PRO; cadenas;  rendimiento  
startswith("  Py"): True
endswith("miento  "): True
find("I/O"): 10
count("a"): 5
join: a | b | c
zfill(5): 00042
center(20, "-"): -------TÍTULO-------


In [19]:

# 15. Formateo: f-strings, str.format, Template
from string import Template

nombre = 'Ana'
nota = 4.75

print(f'Estudiante: {nombre}, nota redondeada: {nota:.1f}')
print('Formato con posiciones: {} -> {:.2f}'.format(nombre, nota))
tpl = Template('Hola, $who. Tu nota es $score.')
print(tpl.substitute(who=nombre, score=nota))


Estudiante: Ana, nota redondeada: 4.8
Formato con posiciones: Ana -> 4.75
Hola, Ana. Tu nota es 4.75.


In [20]:

# 16. Regex: búsqueda y reemplazo, grupos, backreferences
import re

texto = "ID: 2025-10-02 | Usuario: Mario | Email: mario@example.com; Otra: test@test.co"
# Extraer emails
emails = re.findall(r'[\w.\-+]+@[\w\-]+(?:\.[\w\-]+)+', texto)
print('Emails:', emails)

# Reemplazo con grupos nombrados
pat = re.compile(r'(?P<clave>ID):\s*(?P<valor>[\d\-]+)')
m = pat.search(texto)
if m:
    print('Grupo clave:', m.group('clave'), 'valor:', m.group('valor'))

# Normalizar múltiples espacios
s = 'Texto    con   espacios    extras'
s2 = re.sub(r'\s{2,}', ' ', s)
print('Normalizado:', s2)


Emails: ['mario@example.com', 'test@test.co']
Grupo clave: ID valor: 2025-10-02
Normalizado: Texto con espacios extras


In [21]:

# 17. Limpieza/normalización Unicode
import unicodedata

t = "café CAFÉ Straße"
print('Original:', t)

# casefold para comparaciones robustas
print('casefold:', t.casefold())

# Normalización (NFC vs NFD)
nfc = unicodedata.normalize('NFC', t)
nfd = unicodedata.normalize('NFD', t)
print('NFC == NFD?', nfc == nfd)

# Eliminar diacríticos usando NFD + filtro
sin_diac = ''.join(ch for ch in unicodedata.normalize('NFD', t)
                   if unicodedata.category(ch) != 'Mn')
print('Sin diacríticos:', sin_diac)


Original: café CAFÉ Straße
casefold: café café strasse
NFC == NFD? False
Sin diacríticos: cafe CAFE Straße


In [22]:

# 18. str.translate / maketrans: mapeos masivos eficientes
import string

s = "abc123-xyz"
tabla = str.maketrans({c: None for c in string.digits})  # eliminar dígitos
print('Sin dígitos:', s.translate(tabla))

tabla2 = str.maketrans({'-': '_', 'x': 'X'})
print('Reemplazos múltiples:', s.translate(tabla2))


Sin dígitos: abc-xyz
Reemplazos múltiples: abc123_Xyz


In [23]:

# 19. textwrap: envolver y recortar texto
import textwrap

largo = "Python ofrece utilidades para envolver texto en párrafos con ancho fijo, útil para reportes en consola."
print(textwrap.fill(largo, width=40))
print('--- recorte ---')
print(textwrap.shorten(largo, width=50, placeholder='…'))


Python ofrece utilidades para envolver
texto en párrafos con ancho fijo, útil
para reportes en consola.
--- recorte ---
Python ofrece utilidades para envolver texto en…


In [24]:

# 20. Rendimiento práctico
import io

# 1) join vs concatenación en bucles
palabras = [str(i) for i in range(10000)]
# mala práctica: concatenar en bucle (O(n^2) en algunos casos)
s1 = ''
for w in palabras:
    s1 += w  # evitá esto en bucles grandes

# buena práctica
s2 = ''.join(palabras)

print('Longitudes iguales:', len(s1) == len(s2))

# 2) Escritura con buffer explícito
buf = io.StringIO()
for i in range(10000):
    buf.write(str(i))
resultado = buf.getvalue()
print('Buffer OK:', len(resultado) > 0)


Longitudes iguales: True
Buffer OK: True



## 21. Mini‑proyecto: ETL de logs

**Objetivo:** Leer múltiples archivos de log (`.log`), extraer campos con **regex**, normalizar texto, y generar:
- Un **CSV** consolidado.
- Un **JSON** con estadísticas agregadas.

Formato de línea de ejemplo:
```
[2025-10-02 05:43:21] INFO user=ana action=login status=ok ip=10.0.0.1
```


In [25]:
from pathlib import Path
import re, csv, json
from collections import Counter

logs_dir = Path('logs_demo')
logs_dir.mkdir(exist_ok=True)

# Crear logs de ejemplo
plantillas = [
    "[2025-10-02 05:43:21] INFO user=ana action=login status=ok ip=10.0.0.{i}\n",
    "[2025-10-02 05:44:10] WARN user=luis action=download status=fail ip=10.0.0.{i}\n",
    "[2025-10-02 05:45:00] INFO user=zoe action=upload status=ok ip=10.0.0.{i}\n",
]
for idx in range(1, 6):
    p = logs_dir / f'app_{idx}.log'
    with p.open('w', encoding='utf-8') as f:
        for line in plantillas:
            f.write(line.format(i=idx))

# Regex para extraer campos
pat = re.compile(
    r"\[(?P<timestamp>[^\]]+)\]\s+"
    r"(?P<level>INFO|WARN|ERROR)\s+"
    r"user=(?P<user>\w+)\s+"
    r"action=(?P<action>\w+)\s+"
    r"status=(?P<status>\w+)\s+"
    r"ip=(?P<ip>[\d.]+)"
)

# Salidas
csv_out = Path('logs_consolidados.csv')
json_out = Path('logs_stats.json')

registros = []
for archivo in sorted(logs_dir.glob('*.log')):
    for line in archivo.read_text(encoding='utf-8').splitlines():
        m = pat.search(line)
        if m:
            registros.append(m.groupdict())

# Escribir CSV
with csv_out.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp','level','user','action','status','ip'])
    writer.writeheader()
    writer.writerows(registros)

# Estadísticas agregadas
stats = {
    'por_usuario': dict(Counter(r['user'] for r in registros)),
    'por_accion': dict(Counter(r['action'] for r in registros)),
    'por_status': dict(Counter(r['status'] for r in registros)),
    'total': len(registros),
}

json_out.write_text(json.dumps(stats, indent=2, ensure_ascii=False), encoding='utf-8')

print('CSV:', csv_out)
print('JSON stats:', json_out)
print(json.dumps(stats, indent=2, ensure_ascii=False))

CSV: logs_consolidados.csv
JSON stats: logs_stats.json
{
  "por_usuario": {
    "ana": 5,
    "luis": 5,
    "zoe": 5
  },
  "por_accion": {
    "login": 5,
    "download": 5,
    "upload": 5
  },
  "por_status": {
    "ok": 10,
    "fail": 5
  },
  "total": 15
}



## 22. Recursos finales

- Documentación oficial de Python: módulos `io`, `pathlib`, `csv`, `json`, `gzip`, `bz2`, `lzma`, `pickle`, `unicodedata`, `textwrap`, `re`.
- PEP 519 (PathLike), PEP 393 (Flexible String Representation).
- Buenas prácticas: especificar `encoding='utf-8'`, usar context managers, iteración en streaming para archivos grandes, validación de entradas y manejo explícito de errores.
